In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
import random
from dotenv import load_dotenv

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

load_dotenv('../config/.env')  # reuse .env to prevent leak
os.getcwd()

import joblib
import torch
from torch.utils.data import Dataset
from scipy.sparse import csr_matrix

print("CUDA Availiability:", torch.cuda.is_available())
print("CUDA Device:", torch.cuda.get_device_name(0))

CUDA Availiability: True
CUDA Device: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Seed set to {seed}. cuDNN deterministic mode enabled.")

set_seed(SEED)

Seed set to 42. cuDNN deterministic mode enabled.


In [2]:
# Koneksi ke Postgres, baca langsung dari tabel marts hasil dbt
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

#Schema: 'public' fact table from mart
df_raw = pd.read_sql("SELECT item_id, title, price, volume_count, is_ambigous_bulk_pricing, condition, description, text_risk_score FROM public.int_ebay_listing_risk_analysis", engine)

print(f"Total baris: {len(df_raw)}")
df_raw.head()

Total baris: 2525


,item_id,title,price,volume_count,is_ambigous_bulk_pricing,condition,description,text_risk_score
0,v1|115965603147|0,Spy Classroom Volumes 1-3 Light Novel Set New ...,40.5,3.0,False,Brand New,Spy Classroom Novel Volume 1-3 was written by ...,0
1,v1|116204521440|0,"Spy Classroom, Vol. 7 (light novel): A Glint i...",17.2,1.0,False,Brand New,"Spy Classroom, Vol. 7 (light novel): A Glint i...",0
2,v1|116263745717|0,"Gimai Seikatsu, Days with My Step Sister Light...",149.9,8.0,False,Brand New,"Gimai Seikatsu, Days with My Step Sister Light...",70
3,v1|116267961169|0,The Eminence in Shadow 1-6 Novel English New H...,108.0,1.0,False,Brand New,The Eminence in Shadow novel 1-6 (Hardcover) O...,0
4,v1|116268231670|0,"Bofuri: I Don't Want to Get Hurt, so I'll Max ...",17.2,1.0,False,Brand New,"Bofuri: I Don''''t Want to Get Hurt, so I''''l...",0


In [7]:
def compute_embeddings(df: pd.DataFrame, text_col: str = "combined_text", batch_size: int = 64):
    """
    df perlu kolom title & description (atau kolom gabungan custom).
    Menggabungkan title+description jadi satu teks per listing --
    title biasanya lebih padat sinyal (nama series, "reprint", dst),
    description lebih panjang tapi lebih noisy (boilerplate shipping).
    """
    from sentence_transformers import SentenceTransformer
 
    if text_col not in df.columns:
        df = df.copy()
        df[text_col] = (
            df["title"].fillna("") + ". " + df["description"].fillna("")
        ).str.strip()
 
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(
        df[text_col].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,  # supaya cosine similarity = dot product
    )
    return embeddings, df

embeddings, df_all = compute_embeddings(df_raw)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [8]:
def run_hdbscan(embeddings: np.ndarray, min_cluster_size: int = 15, min_samples: int = 5):
    """
    min_cluster_size=15: cluster lebih kecil dari ini dianggap noise --
    disesuaikan untuk dataset ~2500 baris (cluster terlalu kecil akan
    overfit ke kebetulan). Naikkan/turunkan sesuai granularitas yang
    diinginkan setelah lihat hasil pertama.
 
    min_samples=5: kontrol seberapa konservatif HDBSCAN menandai titik
    sebagai noise (-1) vs masuk cluster. Default masuk akal untuk mulai.
    """
    import hdbscan
 
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",  # embedding sudah dinormalisasi, euclidean ~ cosine
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(embeddings)
    return labels, clusterer

In [9]:
def analyze_clusters(df: pd.DataFrame, cluster_labels: np.ndarray, base_rate: float = None):
    """
    Cross-tab cluster vs text_risk_score. Highlight cluster dengan risk-rate
    jauh dari base rate keseluruhan -- ini yang paling informatif.
    """
    df = df.copy()
    df["cluster"] = cluster_labels
    df["is_risk"] = (df["text_risk_score"] > 0).astype(int)
 
    if base_rate is None:
        base_rate = df["is_risk"].mean()
 
    print(f"Base rate risk=1 di seluruh dataset: {base_rate*100:.1f}%")
    print(f"Total cluster ditemukan (exclude noise): {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")
    n_noise = (cluster_labels == -1).sum()
    print(f"Titik noise (tidak masuk cluster manapun): {n_noise} ({n_noise/len(df)*100:.1f}%)")
 
    summary = (
        df.groupby("cluster")
        .agg(n=("is_risk", "size"), risk_rate=("is_risk", "mean"))
        .sort_values("n", ascending=False)
    )
    summary["risk_rate_pct"] = (summary["risk_rate"] * 100).round(1)
    summary["deviation_from_base"] = (summary["risk_rate"] - base_rate).round(3)
 
    print("\n--- Semua cluster, sorted by size ---")
    print(summary[["n", "risk_rate_pct", "deviation_from_base"]].to_string())
 
    print("\n--- Cluster paling informatif (risk-rate PALING JAUH dari base rate, n>=10) ---")
    informative = summary[summary["n"] >= 10].copy()
    informative["abs_dev"] = informative["deviation_from_base"].abs()
    informative = informative.sort_values("abs_dev", ascending=False).head(10)
    print(informative[["n", "risk_rate_pct", "deviation_from_base"]].to_string())
 
    return df, summary

In [10]:
def inspect_cluster_samples(df: pd.DataFrame, cluster_id: int, n_samples: int = 5):
    """Lihat contoh title dari satu cluster untuk interpretasi manual."""
    subset = df[df["cluster"] == cluster_id]
    print(f"\n=== Cluster {cluster_id} (n={len(subset)}, risk_rate={subset['is_risk'].mean()*100:.1f}%) ===")
    sample = subset.sample(min(n_samples, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        risk_flag = "RISK" if row["is_risk"] else "safe"
        print(f"  [{risk_flag}] {row['title'][:100]}")
 
 
def inspect_noise_points(df: pd.DataFrame, n_samples: int = 10):
    """
    Titik noise (cluster=-1) adalah listing yang tidak mirip dengan
    kelompok manapun -- kandidat bagus untuk manual audit karena mereka
    'unik', termasuk kemungkinan kasus seperti 12 baris viz-media yang
    luput dari regex.
    """
    noise = df[df["cluster"] == -1]
    print(f"\n=== Noise points (n={len(noise)}, risk_rate={noise['is_risk'].mean()*100:.1f}%) ===")
    sample = noise.sample(min(n_samples, len(noise)), random_state=42)
    for _, row in sample.iterrows():
        risk_flag = "RISK" if row["is_risk"] else "safe"
        print(f"  [{risk_flag}] {row['title'][:100]}")

In [11]:
cluster_labels, clusterer = run_hdbscan(embeddings, min_cluster_size=15, min_samples=5)

In [12]:
df_clustered, summary = analyze_clusters(df_all, cluster_labels)

Base rate risk=1 di seluruh dataset: 21.4%
Total cluster ditemukan (exclude noise): 33
Titik noise (tidak masuk cluster manapun): 1050 (41.6%)

--- Semua cluster, sorted by size ---
            n  risk_rate_pct  deviation_from_base
cluster                                          
-1       1050           19.5               -0.019
 12       162            0.6               -0.208
 24       144           14.6               -0.068
 19        98           22.4                0.010
 3         98           10.2               -0.112
 7         78            0.0               -0.214
 25        75           14.7               -0.068
 13        67            3.0               -0.184
 9         64           15.6               -0.058
 26        63            1.6               -0.198
 28        56          100.0                0.786
 30        55           16.4               -0.051
 10        44            0.0               -0.214
 31        38           84.2                0.628
 15        33     

In [13]:
inspect_cluster_samples(df_clustered, cluster_id=28, n_samples=8)


=== Cluster 28 (n=56, risk_rate=100.0%) ===
  [RISK] 86 EIGHTY-SIX Light Novel English Version by Asato Volume 1-12 NEW -Fast DHL
  [RISK] 86 EIGHTY-SIX Light Novel LOOSE & Full Set Volumes 1-12 English Version
  [RISK] 86 EIGHTY-SIX Light Novel English Version by Asato Volume 1-13 NEW FAST EXPRESS
  [RISK] 86, EIGHTY-SIX (Light Novel) Vol 1-12 English Version +Free Gift
  [RISK] 86 EIGHTY-SIX Light Novel (SINGLE/MIX/FULL SET) Volume 1-12 English Version Book
  [RISK] 86 EIGHTY-SIX English Version Light Novel Vol.1-13 New By Asato Asato FAST SHIP
  [RISK] 86 EIGHTY-SIX Light Novel English Version by Asato Volume 1-13 NEW EXPEDITED
  [RISK] 86 EIGHTY-SIX Light Novel English Version by Asato Volume 1-13 NEW EXPEDITED


In [14]:
inspect_noise_points(df_clustered, n_samples=15)


=== Noise points (n=1050, risk_rate=19.5%) ===
  [safe] Konosuba God's Blessing on This Wonderful World! Volume 5 (Light Novel)
  [safe] Ascendance of a Bookworm Part 1 Vol.1-6 Light Novel Junior Bunko Easy Japanese
  [RISK] Re:Zero- Starting Life in Another World (Light Novel) Vol 1-26 English Version
  [safe] Sword Art Online 5: Phantom Bullet - light novel - Paperback - GOOD
  [safe] Classroom of the Elite: Year 2, Vol. 7, LIKE NW UNREAD English Light Novel 2023
  [safe] Sword Art Online 7: Mother's Rosary - light novel
  [safe] Bofuri Vol.1-19 Latest Full Set Japanese Light Novel Japan JPN
  [safe] Sword Art Online Light Novel Volume 1-28 Set (KADOKAWA)
  [RISK] Mushoku Tensei: Jobless Reincarnation (Light Novel) Vol. 3 Multicolor
  [safe] Re Zero Starting Life in Another World Light Novel 24, Paperback by Nagatsuki...
  [RISK] Re:Zero: Starting Life in Another World Light Novel New Vol 1-26 English Version
  [safe] Spy Classroom Grete Beloved Daughter Light Novel  2 Pages
  [safe

In [15]:
def inspect_pure_clusters(df: pd.DataFrame, summary: pd.DataFrame, target_rate: float = 0.0,
                           min_n: int = 10, n_samples_per_cluster: int = 8):
    """
    Inspect semua cluster dengan risk_rate yang sama persis dengan
    target_rate (0.0 untuk cluster 100%-safe, 1.0 untuk cluster 100%-risk),
    n>=min_n. Print sample title dari tiap cluster supaya bisa dibaca
    manual untuk cari pola false-negative/false-positive tersembunyi --
    misal listing yang secara semantik dekat dengan cluster risk=100%
    tapi entah kenapa masuk cluster berbeda dengan label 0%, atau
    sebaliknya.
 
    target_rate=0.0: cari FALSE NEGATIVE (harusnya risk, dilabel safe)
    target_rate=1.0: cari FALSE POSITIVE (harusnya safe, dilabel risk)
    """
    matching = summary[(summary["risk_rate"] == target_rate) & (summary["n"] >= min_n)]
    print(f"Cluster dengan risk_rate == {target_rate} dan n >= {min_n}: {len(matching)} cluster")
    print(f"Total baris across cluster ini: {matching['n'].sum()}\n")
 
    for cluster_id in matching.index:
        subset = df[df["cluster"] == cluster_id]
        print(f"=== Cluster {cluster_id} (n={len(subset)}, risk_rate={target_rate*100:.0f}%) ===")
        sample = subset.sample(min(n_samples_per_cluster, len(subset)), random_state=42)
        for _, row in sample.iterrows():
            print(f"  {row['title'][:110]}")
        print()
 
    return matching

In [16]:
inspect_pure_clusters(df_clustered, summary, target_rate=0.0, min_n=10, n_samples_per_cluster=8)

Cluster dengan risk_rate == 0.0 dan n >= 10: 9 cluster
Total baris across cluster ini: 276

=== Cluster 7 (n=78, risk_rate=0%) ===
  Overlord, Vol. 7 (Light Novel): The Invaders of the Great Tomb Volume 7: New
  Overlord, Vol. 6 (light Novel) : The Men of the Kingdom Part II by Kugane ...
  Overlord, Vol. 8 (Light Novel): The Two Leaders Volume 8 by Kugane Maruyama: New
  Overlord Vol 1-9 Hardcover Light Novel Set Kugane Maruyama
  Overlord, Vol. 4 (light novel): The Lizardman Heroes (Volume 4)
  Overlord Light Novel Volumes 1-14
  Overlord, Vol. 4 (light novel): The Lizardman Heroes - Hardcover - GOOD
  Overlord, Vol. 1 - light novel

=== Cluster 10 (n=44, risk_rate=0%) ===
  Bofuri: I Don't Want to Get Hurt, So I'll Max Out My Defense., Vol. 3 (Light
  Bofuri: I Don't Want to Get Hurt, so I'll Max Out My Defense., Vol. 4 (light nov
  Bofuri: I Don't Want to Get Hurt, so I'll Max Out My Defense., Vol. 8 (light nov
  Bofuri: I Don't Want to Get Hurt, So I'll Max Out My Defense., Vol. 4

,n,risk_rate,risk_rate_pct,deviation_from_base
cluster,,,,
7,78,0.0,0.0,-0.214
10,44,0.0,0.0,-0.214
18,32,0.0,0.0,-0.214
0,27,0.0,0.0,-0.214
11,22,0.0,0.0,-0.214
16,21,0.0,0.0,-0.214
20,20,0.0,0.0,-0.214
5,17,0.0,0.0,-0.214
27,15,0.0,0.0,-0.214


In [17]:
def cross_check_with_source(df: pd.DataFrame, item_id_col: str = "item_id"):
    """
    Cross-check hasil clustering dengan text_risk_score LANGSUNG dari
    df_clustered (yang sudah include text_risk_score dari kolom asli),
    plus tambahan: title, description mentah, dan regex match manual
    per baris -- supaya bisa audit satu-satu tanpa perlu balik ke SQL.
 
    Menghasilkan DataFrame siap-export ke CSV untuk audit manual di luar
    notebook (mis. buka di Excel/DBeaver untuk verifikasi ground truth
    per baris), dan juga highlight baris yang PALING layak diprioritaskan
    untuk audit manual (title mengandung indikator full-set/express/loose
    TAPI text_risk_score=0).
 
    Regex pattern di bawah ini SAMA PERSIS dengan yang ada di
    int_ebay_listing_risk_analysis.sql -- supaya kita tahu baris mana
    yang genuinely tidak match regex (bukan salah copy pattern).
    """
    import re
 
    # Pattern identik dengan SQL asli (title OR description OR condition IS NULL)
    title_pattern = re.compile(r'(reprint|unbranded|bootleg|pdf|ebook|custom print|reading edition)', re.I)
    desc_pattern = re.compile(
        r'(reprint|not original|loose|print on demand|not an official|not official|'
        r'not suitable|fan translated|fan-translated|not from original|reading edition)', re.I
    )
    negation_desc = re.compile(
        r"(not|no|never|isn't|isnt|without)\s+(\w+\s+){0,4}(reprint|unofficial|fan.?translated|bootleg)", re.I
    )
    negation_title = re.compile(r"(not|no|never)\s+(\w+\s+){0,4}(reprint|unofficial|bootleg)", re.I)
 
    def recompute_regex_flag(row):
        title = row.get("title", "")
        desc = row.get("description", "")
        title = "" if pd.isna(title) else str(title)
        desc = "" if pd.isna(desc) else str(desc)
        condition = row.get("condition", None)
 
        raw_match = bool(
            title_pattern.search(title) or desc_pattern.search(desc) or pd.isna(condition)
        )
        negated = bool(negation_desc.search(desc) or negation_title.search(title))
        return 70 if (raw_match and not negated) else 0
 
    df = df.copy()
    if "condition" in df.columns:
        df["recomputed_risk_score"] = df.apply(recompute_regex_flag, axis=1)
        mismatch = df["recomputed_risk_score"] != df["text_risk_score"]
        print(f"Baris di mana recomputed regex TIDAK match text_risk_score asli: {mismatch.sum()}")
        if mismatch.sum() > 0:
            print("(Ini indikasi kolom 'condition' di df tidak sinkron dengan yang dipakai SQL asli,")
            print(" atau ada perbedaan lain -- cek dulu sebelum percaya hasil manual matching di bawah)")
    else:
        print("Kolom 'condition' tidak ada di df -- skip recompute, pakai text_risk_score asli saja.")
 
    # Manual "should probably be risky" heuristic -- BUKAN regex resmi,
    # cuma untuk highlight baris yang layak diprioritaskan audit manual
    suspicious_phrases = re.compile(
        r'(full set|loose|dhl express|fast ship|express ship|premium leather|royal edition)', re.I
    )
    df["title_has_suspicious_phrase"] = df["title"].fillna("").astype(str).apply(lambda t: bool(suspicious_phrases.search(t)))
 
    priority_audit = df[
        (df["text_risk_score"] == 0) & (df["title_has_suspicious_phrase"])
    ].copy()
 
    print(f"\nBaris berlabel text_risk_score=0 TAPI title mengandung frasa mencurigakan "
          f"(full set/loose/DHL express/dst): {len(priority_audit)}")
    print("Ini kandidat prioritas untuk manual audit ground truth -- SAMA seperti proses")
    print("yang dipakai untuk temukan 12 baris viz-media kemarin, tapi sekarang otomatis")
    print("di-scan across seluruh dataset, bukan cuma satu term.\n")
 
    if len(priority_audit) > 0:
        print(priority_audit[["title", "cluster", "text_risk_score"]].head(20).to_string())
 
    return df, priority_audit

In [18]:
def export_for_manual_audit(priority_audit: pd.DataFrame, path: str = "/mnt/user-data/outputs/priority_audit.csv"):
    """Export kandidat prioritas audit ke CSV supaya bisa dibuka di Excel/
    DBeaver untuk verifikasi ground truth satu-satu, tanpa perlu balik ke
    notebook tiap kali."""
    cols = [c for c in ["item_id", "title", "description", "cluster", "text_risk_score"] if c in priority_audit.columns]
    # priority_audit[cols].to_csv(path, index=False)
    # print(f"Exported {len(priority_audit)} baris ke {path}")

In [19]:
df_checked, priority_audit = cross_check_with_source(df_clustered)
export_for_manual_audit(priority_audit)

Baris di mana recomputed regex TIDAK match text_risk_score asli: 0

Baris berlabel text_risk_score=0 TAPI title mengandung frasa mencurigakan (full set/loose/DHL express/dst): 97
Ini kandidat prioritas untuk manual audit ground truth -- SAMA seperti proses
yang dipakai untuk temukan 12 baris viz-media kemarin, tapi sekarang otomatis
di-scan across seluruh dataset, bukan cuma satu term.

                                                                                title  cluster  text_risk_score
39     Mushoku Tensei Jobless Reincarnation Vol.1-26 Complete Full Set Light Novel JP       -1                0
48                               Bofuri Vol.1-19 Latest Full Set Japanese Light Novel       -1                0
107   Too Many Losing Heroines Light Novel Vol 1 - Vol 6 English Version Loose / Full        2                0
225     Konosuba God's Blessing on This Wonderful World Vol.1-17 Full set Light Novel       26                0
229                             Bofuri Vol.1-18 La

In [20]:
import re
pattern_royal_premium = re.compile(
    r'\[?\s*(royal edition|premium leather bound)\s*\]?', re.I
)

pattern_full_set = re.compile(r'full\s*set', re.I)
pattern_vol_range = re.compile(
    r'(?:vol\.?|vols\.?|volume|volumes)\s*(\d+)\s*-\s*(\d+)', re.I
)



In [21]:
def classify_patterns(df: pd.DataFrame, wide_range_threshold: int = 10) -> pd.DataFrame:
    df = df.copy()
    title_str = df["title"].fillna("").astype(str)
 
    df["pattern_royal_premium"] = title_str.apply(lambda t: bool(pattern_royal_premium.search(t)))
 
    def is_wide_range_full_set(title: str) -> bool:
        if not pattern_full_set.search(title):
            return False
        match = pattern_vol_range.search(title)
        if not match:
            return False
        start, end = int(match.group(1)), int(match.group(2))
        return (end - start) >= wide_range_threshold
 
    df["pattern_full_set_wide_range"] = title_str.apply(is_wide_range_full_set)
 
    return df
 
 
def summarize_pattern(df: pd.DataFrame, pattern_col: str, pattern_label: str):
    subset = df[df[pattern_col]]
    n_total = len(subset)
    if n_total == 0:
        print(f"{pattern_label}: 0 baris ditemukan di dataset.")
        return
 
    n_flagged = (subset["text_risk_score"] > 0).sum()
    n_unflagged = n_total - n_flagged
    pct_unflagged = n_unflagged / n_total * 100
 
    print(f"=== {pattern_label} ===")
    print(f"Total baris dengan pola ini      : {n_total}")
    print(f"Sudah ter-flag text_risk_score=70: {n_flagged} ({n_flagged/n_total*100:.1f}%)")
    print(f"BELUM ter-flag (text_risk_score=0): {n_unflagged} ({pct_unflagged:.1f}%)  <-- estimasi false-negative")
    print()
 
 
def run_full_summary(df: pd.DataFrame, wide_range_threshold: int = 10):
    df = classify_patterns(df, wide_range_threshold=wide_range_threshold)
 
    print(f"Total dataset: {len(df)} baris\n")
    summarize_pattern(df, "pattern_royal_premium", "Pola B: [Royal Edition] / [Premium Leather Bound]")
    summarize_pattern(df, "pattern_full_set_wide_range", f"Pola A: 'Full Set' + volume range >= {wide_range_threshold}")
 
    # Overlap check -- kalau dua pola ini sering muncul bareng di baris
    # yang sama, itu juga informasi berguna (satu jenis seller yang pakai
    # kedua trik sekaligus)
    both = df[df["pattern_royal_premium"] & df["pattern_full_set_wide_range"]]
    print(f"Baris yang match KEDUA pola sekaligus: {len(both)}")
 
    return df

In [22]:
df_all_patterns = run_full_summary(df_all, wide_range_threshold=10)

Total dataset: 2525 baris

=== Pola B: [Royal Edition] / [Premium Leather Bound] ===
Total baris dengan pola ini      : 46
Sudah ter-flag text_risk_score=70: 24 (52.2%)
BELUM ter-flag (text_risk_score=0): 22 (47.8%)  <-- estimasi false-negative

=== Pola A: 'Full Set' + volume range >= 10 ===
Total baris dengan pola ini      : 170
Sudah ter-flag text_risk_score=70: 120 (70.6%)
BELUM ter-flag (text_risk_score=0): 50 (29.4%)  <-- estimasi false-negative

Baris yang match KEDUA pola sekaligus: 0


In [23]:
df_all_patterns[df_all_patterns['pattern_royal_premium'] & (df_all_patterns['text_risk_score']==0)][['title','text_risk_score']].sample(10, random_state=42)
df_all_patterns[df_all_patterns['pattern_full_set_wide_range'] & (df_all_patterns['text_risk_score']==0)][['title','text_risk_score']].sample(10, random_state=42)

,title,text_risk_score
510,Bofuri Vol.1-19 Latest Full Set Japanese Light...,0
1948,Spy Classroom Vol.1-12 + 4 short stories Lates...,0
1913,Mushoku Tensei Vol.1-26 ＋ Extra 1-2 ＋ Special ...,0
2228,Konosuba God's Blessing on This Wonderful Worl...,0
1178,86 EIGHTY-SIX Light Novel English Version Vol....,0
2315,86 EIGHTY-SIX Light Novel (SINGLE/MIX/FULL SET...,0
1719,[Japanese Language Novel] 86: Eighty Six Vol.1...,0
1709,Sword Art Online Vol.1-27 Latest Full Set Japa...,0
1916,Classroom of the Elite Vol.1-11.5 ＋Year 2 Vol....,0
1200,Re:Zero-Starting Life in Another World Full Se...,0


In [30]:
def cross_check_price_signal(df: pd.DataFrame, pattern_col: str = "pattern_full_set_wide_range"):
    """
    df perlu kolom: title, text_risk_score, price, volume_count,
    is_ambiguous_bulk_pricing, dan pattern_col (dari classify_patterns()
    di measure_pattern_scale.py -- jalankan itu dulu kalau belum ada).
    """
    required_cols = ["price", "volume_count", "is_ambigous_bulk_pricing", pattern_col, "text_risk_score"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Kolom hilang: {missing}. Pastikan df_all di-query ulang dari "
            f"int_ebay_listing_risk_analysis (bukan cuma dari RiskDataLoader "
            f"yang mungkin hanya select title/description/text_risk_score), "
            f"dan sudah dijalankan classify_patterns() dari measure_pattern_scale.py."
        )
 
    df = df.copy()
    df["price_per_volume"] = df["price"] / df["volume_count"].replace(0, np.nan)
 
    subset = df[df[pattern_col]].copy()
    unflagged = subset[subset["text_risk_score"] == 0]
    flagged = subset[subset["text_risk_score"] > 0]
 
    print(f"Total baris Pola A: {len(subset)}")
    print(f"  - Sudah ter-flag text_risk_score=70 : {len(flagged)}")
    print(f"  - Belum ter-flag (text_risk_score=0) : {len(unflagged)}\n")
 
    print("--- price_per_volume: dibandingkan antara flagged vs unflagged ---")
    print(f"Flagged   (n={len(flagged)}): median=${flagged['price_per_volume'].median():.2f}, "
          f"mean=${flagged['price_per_volume'].mean():.2f}")
    print(f"Unflagged (n={len(unflagged)}): median=${unflagged['price_per_volume'].median():.2f}, "
          f"mean=${unflagged['price_per_volume'].mean():.2f}\n")
 
    print("--- is_ambiguous_bulk_pricing overlap dengan Pola A yang UNFLAGGED ---")
    n_ambiguous = unflagged["is_ambigous_bulk_pricing"].sum()
    print(f"Dari {len(unflagged)} baris Pola A yang belum ter-flag text_risk_score,")
    print(f"{n_ambiguous} ({n_ambiguous/len(unflagged)*100:.1f}%) juga ter-flag is_ambigous_bulk_pricing.")
    print("(Jika tinggi -> sinyal harga MENDUKUNG dugaan bootleg/reprint pada baris ini)")
    print("(Jika rendah -> harga per-volume tampak wajar, dugaan bootleg dari keyword saja kurang kuat)\n")
 
    # Breakdown per-series untuk lihat variasi -- kemungkinan tidak semua
    # series di Pola A punya pola harga sama (misal 86 Eighty-Six vs
    # Re:Zero bisa beda karakteristik pasar)
    print("--- price_per_volume median per judul (unflagged Pola A only, top 15 by count) ---")
    # Ambil "nama seri" kasar dari title (kata-kata sebelum 'Vol')
    unflagged["series_guess"] = unflagged["title"].str.extract(r'^(.*?)(?:vol\.?|volume)', flags=2, expand=False)
    unflagged["series_guess"] = unflagged["series_guess"].fillna(unflagged["title"]).str.strip()
 
    series_summary = (
        unflagged.groupby("series_guess")
        .agg(n=("price_per_volume", "size"), median_ppv=("price_per_volume", "median"))
        .sort_values("n", ascending=False)
        .head(15)
    )
    print(series_summary.to_string())
 
    return subset, unflagged, flagged
 
 
def sample_for_manual_check(unflagged: pd.DataFrame, n: int = 15, sort_by_price: bool = True):
    """
    Ambil sample untuk audit manual -- diurutkan by price_per_volume
    (termurah dulu) supaya kandidat paling mencurigakan (harga per-volume
    sangat rendah) diperiksa lebih dulu, bukan random sample murni.
    """
    cols = [c for c in ["item_id", "title", "price", "volume_count", "price_per_volume",
                          "is_ambigous_bulk_pricing", "text_risk_score"] if c in unflagged.columns]
    if sort_by_price:
        result = unflagged.sort_values("price_per_volume").head(n)
    else:
        result = unflagged.sample(min(n, len(unflagged)), random_state=42)
    print(result[cols].to_string())
    return result

In [31]:
df_all = classify_patterns(df_all, wide_range_threshold=10)
subset, unflagged, flagged = cross_check_price_signal(df_all)

Total baris Pola A: 170
  - Sudah ter-flag text_risk_score=70 : 120
  - Belum ter-flag (text_risk_score=0) : 50

--- price_per_volume: dibandingkan antara flagged vs unflagged ---
Flagged   (n=120): median=$12.87, mean=$12.40
Unflagged (n=50): median=$9.07, mean=$9.94

--- is_ambiguous_bulk_pricing overlap dengan Pola A yang UNFLAGGED ---
Dari 50 baris Pola A yang belum ter-flag text_risk_score,
1 (2.0%) juga ter-flag is_ambigous_bulk_pricing.
(Jika tinggi -> sinyal harga MENDUKUNG dugaan bootleg/reprint pada baris ini)
(Jika rendah -> harga per-volume tampak wajar, dugaan bootleg dari keyword saja kurang kuat)

--- price_per_volume median per judul (unflagged Pola A only, top 15 by count) ---
                                                          n  median_ppv
series_guess                                                           
Konosuba God's Blessing on This Wonderful World           6    5.117647
Overlord                                                  6    8.109063
Bofuri   